In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_FILE  = "messages_week.xlsx"   # твой файл с сообщениями за неделю
SHEET       = None                   # None = первый лист
OUTPUT_FILE = "providers_found.xlsx"
GLOSSARY_DB = "PSP_provider_glossary_builder/data/provider_glossary.db"

MIN_TEXT_LEN   = 20
REQUEST_DELAY  = 0.3

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import sys, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('../.env'))

PROJECT_ROOT = str(Path('..').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from glossary_builder.llm import LLMClient
from PSP_provider_glossary_builder.provider_classifier import (
    classify_provider_message,
    load_provider_glossary,
)
from PSP_provider_glossary_builder.provider_decision import ProviderExtractionConfig

print('OK')

In [ ]:
# ── ЗАГРУЗКА + ДЕДУПЛИКАЦИЯ ───────────────────────────────────────────────────
if INPUT_FILE.endswith('.csv'):
    raw = pd.read_csv(INPUT_FILE)
else:
    raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET)

raw['text']     = raw['text'].fillna('').astype(str).str.strip()
raw['username'] = raw['username'].fillna('').astype(str).str.strip() \
    if 'username' in raw.columns else ''

print(f'Загружено:         {len(raw):>8,} сообщений')

# Шаг 1 — убираем короткие
df = raw[raw['text'].str.len() >= MIN_TEXT_LEN].copy()
print(f'После фильтра <{MIN_TEXT_LEN}: {len(df):>8,}')

# Шаг 2 — дедупликация по тексту
before = len(df)
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'После деdup текст: {len(df):>8,}  (убрано {before - len(df):,})')

# Шаг 3 — дедупликация по username + text (убираем одного юзера с одним сообщением)
if 'username' in df.columns and df['username'].any():
    before = len(df)
    df = df.drop_duplicates(subset=['username', 'text']).reset_index(drop=True)
    print(f'После деdup user:  {len(df):>8,}  (убрано {before - len(df):,})')

print(f'\nИтого для прогона: {len(df):,} сообщений')

In [ ]:
# ── ПРОГОН ───────────────────────────────────────────────────────────────────
glossary = load_provider_glossary(GLOSSARY_DB)
llm      = LLMClient()
cfg      = ProviderExtractionConfig(pre_filter_db_path=GLOSSARY_DB)

print(f'Глоссарий: {len(glossary)} терминов')
print(f'LLM: {llm.provider} / {llm.model}')
print(f'Запуск на {len(df):,} сообщениях...\n')

results = []
total   = len(df)

for n, (_, row) in enumerate(df.iterrows(), 1):
    res = classify_provider_message(
        text       = row['text'],
        glossary   = glossary,
        llm        = llm,
        cfg        = cfg,
        message_id = row.get('message_id'),
        username   = row.get('username'),
    )
    # Добавляем оригинальные поля для контекста в Excel
    res['date']     = row.get('date', '')
    res['group_id'] = row.get('group_id', '')
    results.append(res)

    if n % 50 == 0 or n == total:
        found = sum(1 for r in results if r.get('is_provider'))
        pct   = found / n * 100
        print(f'  [{n:>5}/{total}]  провайдеров: {found}  ({pct:.1f}%)')

    time.sleep(REQUEST_DELAY)

results_df = pd.DataFrame(results)
print(f'\nПрогон завершён.')

In [ ]:
# ── ИТОГИ + СТОИМОСТЬ ─────────────────────────────────────────────────────────
providers = results_df[results_df['is_provider'] == True]
total_msg = len(results_df)
found     = len(providers)

print('=' * 50)
print('  РЕЗУЛЬТАТЫ')
print('=' * 50)
print(f'  Всего сообщений прогнано : {total_msg:,}')
print(f'  Провайдеров найдено      : {found:,}  ({found/total_msg*100:.1f}%)')
print(f'  Не провайдеры            : {total_msg - found:,}')

if found:
    print(f'\n  Уверенность (avg)        : {providers["confidence"].mean():.2f}')
    print(f'  Уверенность (min)        : {providers["confidence"].min():.2f}')

    # Топ вертикалей
    from collections import Counter
    verticals = Counter()
    for v in providers['vertical']:
        for item in str(v).split(','):
            item = item.strip()
            if item and item not in ('', 'nan'):
                verticals[item] += 1
    if verticals:
        print(f'\n  Топ вертикалей:')
        for v, cnt in verticals.most_common(5):
            print(f'    {v:<20} {cnt}')

    # Топ GEO
    geos = Counter()
    for g in providers['geo']:
        for item in str(g).split(','):
            item = item.strip()
            if item and item not in ('', 'nan'):
                geos[item] += 1
    if geos:
        print(f'\n  Топ GEO:')
        for g, cnt in geos.most_common(5):
            print(f'    {g:<20} {cnt}')

# Стоимость запуска
print()
print('=' * 50)
print('  СТОИМОСТЬ')
print('=' * 50)
usage = llm.usage.to_dict()
print(f'  LLM вызовов              : {usage["calls"]:,}')
print(f'  Токенов входящих         : {usage["input_tokens"]:,}')
print(f'  Токенов исходящих        : {usage["output_tokens"]:,}')
print(f'  Токенов всего            : {usage["input_tokens"] + usage["output_tokens"]:,}')
print(f'  Стоимость (est.)         : ${usage["estimated_cost_usd"]:.4f}')
print(f'  Стоимость на сообщение   : ${usage["estimated_cost_usd"] / max(total_msg, 1):.6f}')

In [ ]:
# ── ЭКСПОРТ В EXCEL ───────────────────────────────────────────────────────────
# Лист 1: только провайдеры
# Лист 2: все результаты
# Лист 3: сводка + стоимость

PROVIDER_COLS = [
    'message_id', 'username', 'date',
    'is_provider', 'confidence',
    'company', 'geo', 'methods', 'vertical',
    'evidence_quote', 'rationale', 'text',
]
ALL_COLS = PROVIDER_COLS + ['glossary_terms_seen', 'elapsed_ms']

GREEN  = PatternFill('solid', fgColor='D6F0D6')
HDR    = PatternFill('solid', fgColor='1A3A5C')
GRAY   = PatternFill('solid', fgColor='F2F2F2')
THIN   = Border(
    left=Side(style='thin', color='CCCCCC'), right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),  bottom=Side(style='thin', color='CCCCCC'),
)
WIDTHS = {
    'message_id': 12, 'username': 18, 'date': 18,
    'is_provider': 12, 'confidence': 12,
    'company': 25, 'geo': 20, 'methods': 25, 'vertical': 20,
    'evidence_quote': 50, 'rationale': 55, 'text': 80,
    'glossary_terms_seen': 40, 'elapsed_ms': 12,
}

def write_sheet(ws, df_sheet, cols, row_fill_fn):
    actual = [c for c in cols if c in df_sheet.columns]
    for ci, col in enumerate(actual, 1):
        cell = ws.cell(row=1, column=ci, value=col)
        cell.fill = HDR
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center')
        cell.border = THIN
        ws.column_dimensions[cell.column_letter].width = WIDTHS.get(col, 15)
    for ri, (_, row) in enumerate(df_sheet[actual].iterrows(), 2):
        fill = row_fill_fn(row)
        for ci, col in enumerate(actual, 1):
            val  = row[col]
            cell = ws.cell(row=ri, column=ci, value=str(val) if val is not None else '')
            cell.fill = fill
            cell.border = THIN
            cell.alignment = Alignment(
                wrap_text=(col in ('text', 'rationale', 'evidence_quote')),
                vertical='top',
            )
        ws.row_dimensions[ri].height = 55
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f"A1:{ws.cell(1, len(actual)).column_letter}1"

wb = openpyxl.Workbook()

# Лист 1 — провайдеры
ws1 = wb.active
ws1.title = 'providers'
write_sheet(ws1, providers, PROVIDER_COLS, lambda r: GREEN)

# Лист 2 — все результаты
ws2 = wb.create_sheet('all_results')
write_sheet(
    ws2, results_df, ALL_COLS,
    lambda r: GREEN if r.get('is_provider') else GRAY
)

# Лист 3 — сводка
ws3 = wb.create_sheet('summary')
summary_rows = [
    ('Метрика',                    'Значение'),
    ('Сообщений прогнано',         total_msg),
    ('Провайдеров найдено',        found),
    ('% от прогнанных',           f'{found/total_msg*100:.1f}%'),
    ('Уверенность avg',           f'{providers["confidence"].mean():.2f}' if found else '—'),
    ('',                           ''),
    ('LLM вызовов',               usage['calls']),
    ('Токенов входящих',          usage['input_tokens']),
    ('Токенов исходящих',         usage['output_tokens']),
    ('Стоимость (est. USD)',       f'${usage["estimated_cost_usd"]:.4f}'),
    ('Стоимость на сообщение',    f'${usage["estimated_cost_usd"] / max(total_msg, 1):.6f}'),
    ('',                           ''),
    ('Модель',                     llm.model),
    ('Глоссарий терминов',         len(glossary)),
]
for r, (a, b) in enumerate(summary_rows, 1):
    ca = ws3.cell(row=r, column=1, value=a)
    cb = ws3.cell(row=r, column=2, value=b)
    ca.font = Font(bold=(r == 1))
ws3.column_dimensions['A'].width = 30
ws3.column_dimensions['B'].width = 20

wb.save(OUTPUT_FILE)
print(f'Сохранено → {OUTPUT_FILE}')
print(f'  Лист "providers"   : {found} провайдеров')
print(f'  Лист "all_results" : {total_msg} всего')
print(f'  Лист "summary"     : сводка и стоимость')